# Single-cell wrangling cookbook

## Contents

- [Load an annotated object](#load-an-annotated-object)
- [Attach donor metadata](#attach-donor-metadata)
- [Filter a cohort](#filter-a-cohort)
- [Select an analysis schema](#select-an-analysis-schema)
- [Order, derive, and summarize](#order-derive-and-summarize)
- [Reuse groups](#reuse-groups)
- [Build tidy tables](#build-tidy-tables)
- [Compose with project code](#compose-with-project-code)

A broad tour of the verb surface, run against Scanpy's PBMC3K: 2,638 cells,
1,838 highly variable genes in `X`, Louvain cell-type labels, QC metrics,
PCA/UMAP/t-SNE embeddings, PCA loadings in `varm`, and log-normalised counts
for 13,714 genes in `.raw`.

After `ap.tidy(adata)`, verbs are methods and chain directly. Each container has
a name in `annplyr.data`: `obs`, `var`, `X`, `raw`, and any layer or `obsm` key
such as `X_umap`. Every verb returns a new object. Grouped verbs keep their
grouping until `ungroup()`. Summaries and extraction make an explicit transition
to pandas.

```{note}
`X` holds **scaled** values in this dataset, so expression examples read `raw`.
Only the `donor_id` column is synthetic; it is created where it first appears.
```

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata",
#     "annplyr",
#     "numpy",
#     "pandas",
#     "scanpy",
# ]
# ///

## Load an annotated object

In [2]:
import pandas as pd
import scanpy as sc

import annplyr as ap
from annplyr.data import PCs, X, X_pca, X_umap, obs, raw, var

adata = ap.tidy(sc.datasets.pbmc3k_processed())
adata

AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

`ap.tidy` wraps the object so the verbs are methods. Importing annplyr also
registers the `adata.ap` accessor, which works on a plain `AnnData`.

In [3]:
hasattr(adata, "ap")

True

## Attach donor metadata

In [4]:
adata.obs["donor_id"] = [f"donor_{i % 3 + 1}" for i in range(adata.n_obs)]

donor_table = pd.DataFrame(
    {
        "donor_id": ["donor_1", "donor_2", "donor_3"],
        "condition": ["control", "stimulated", "stimulated"],
    }
)

joined = adata.left_join(donor_table, by="donor_id", relationship="many-to-one")
joined.obs[["louvain", "donor_id", "condition"]].head()

,louvain,donor_id,condition
index,,,
AAACATACAACCAC-1,CD4 T cells,donor_1,control
AAACATTGAGCTAC-1,B cells,donor_2,stimulated
AAACATTGATCAGC-1,CD4 T cells,donor_3,stimulated
AAACCGTGCTTCCG-1,CD14+ Monocytes,donor_1,control
AAACCGTGTATGCG-1,NK cells,donor_2,stimulated


## Filter a cohort

In [5]:
adata.filter(obs.louvain == "B cells")

AnnData object with n_obs × n_vars = 342 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

This is equivalent to standard AnnData indexing:

In [6]:
adata[adata.obs["louvain"] == "B cells", :]

View of AnnData object with n_obs × n_vars = 342 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

Expressions combine with `&` and `|`; predicates passed as separate arguments apply as an implicit `AND`.

In [7]:
adata.filter(
    obs.louvain == "B cells",
    obs.n_genes >= 800,
)

AnnData object with n_obs × n_vars = 85 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

In [8]:
adata.filter((obs.louvain.is_in(["NK cells", "CD8 T cells"])) | (obs.n_counts > 6000))

AnnData object with n_obs × n_vars = 491 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

Filtering on `var` subsets features.

In [9]:
adata.filter(var.n_cells >= 1500)

AnnData object with n_obs × n_vars = 2638 × 21
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

Filtering on `raw` reads expression by gene name. Use `X` for the main matrix, or a layer imported from `annplyr.data`, such as `counts`, for a named layer.

In [10]:
adata.filter(raw.MS4A1 > 2)

AnnData object with n_obs × n_vars = 9 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

Sources can be combined in one call.

In [11]:
adata.filter(
    obs.percent_mito < 0.03,
    var.n_cells >= 500,
    raw.CD79A > 0,
)

AnnData object with n_obs × n_vars = 359 × 208
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

`obs_names` and `var_names` match against axis labels themselves.

In [12]:
adata.filter(ap.obs_names.str.starts_with("AAAC"))

AnnData object with n_obs × n_vars = 12 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

The tidyselect-aware `if_any` and `if_all` helpers reduce a predicate across selected columns.

In [13]:
adata.filter(ap.if_any(ap.starts_with("n_"), lambda column: ap.col(column) > 5000))

AnnData object with n_obs × n_vars = 64 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'donor_id'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

## Select an analysis schema

In [14]:
adata.select(
    obs.louvain,
    obs.n_genes,
    var.n_cells,
    X.MS4A1,
    X.CD79A,
    X.NKG7,
)

AnnData object with n_obs × n_vars = 2638 × 3
    obs: 'louvain', 'n_genes'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

Tidyselect-style helpers make common selections compact. A minus sign drops a column.

In [15]:
adata.select(ap.starts_with("n_"), -obs.n_counts)

AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

`pick` wraps a selector so it can sit among other names.

In [16]:
adata.to_wide(ap.pick(ap.starts_with("n_"))).head()

,n_genes,n_counts
index,,
AAACATACAACCAC-1,781,2419.0
AAACATTGAGCTAC-1,1352,4903.0
AAACATTGATCAGC-1,1131,3147.0
AAACCGTGCTTCCG-1,960,2639.0
AAACCGTGTATGCG-1,522,980.0


## Order, derive, and summarize

In [17]:
adata.arrange(ap.desc(obs.n_counts)).obs[["louvain", "n_counts"]].head()

,louvain,n_counts
index,,
ACGAACTGGCTATG-1,Megakaryocytes,8875.0
GGGCCAACCTTGGA-1,Dendritic cells,8415.0
CAGGTTGAGGATCT-1,B cells,8011.0
ACGAGGGACAGGAG-1,Dendritic cells,7928.0
CATACTTGGGTTAC-1,CD4 T cells,7167.0


In [18]:
adata.slice_head(n=3).obs[["louvain", "n_counts"]]

,louvain,n_counts
index,,
AAACATACAACCAC-1,CD4 T cells,2419.0
AAACATTGAGCTAC-1,B cells,4903.0
AAACATTGATCAGC-1,CD4 T cells,3147.0


In [19]:
adata.slice_max(var.n_cells, n=5, axis="var").var

,n_cells
index,
CYBA,2277
S100A6,2109
UBB,2094
CD37,2063
ARPC1B,2004


`mutate` writes new `obs` metadata, or `var` metadata with `into="var"`. Matrix, layer, `raw`, `obsm`, and `varm` sources are read-only.

In [20]:
adata.mutate(
    deep=obs.n_counts >= 2000,
    b_markers=(raw.MS4A1 > 0) & (raw.CD79A > 0),
    umap1=X_umap[0],
).obs.head()

,n_genes,percent_mito,n_counts,louvain,donor_id,deep,b_markers,umap1
index,,,,,,,,
AAACATACAACCAC-1,781,0.030178,2419.0,CD4 T cells,donor_1,True,False,1.352856
AAACATTGAGCTAC-1,1352,0.037936,4903.0,B cells,donor_2,True,True,-0.478024
AAACATTGATCAGC-1,1131,0.008897,3147.0,CD4 T cells,donor_3,True,False,2.165889
AAACCGTGCTTCCG-1,960,0.017431,2639.0,CD14+ Monocytes,donor_1,True,False,-8.695493
AAACCGTGTATGCG-1,522,0.012245,980.0,NK cells,donor_2,False,False,2.065218


`across` expands one expression over selected columns.

In [21]:
adata.mutate(
    scaled=ap.across(ap.starts_with("n_"), lambda name: ap.col(name) / 1000, names="{col}_k"),
).obs.head()

,n_genes,percent_mito,n_counts,louvain,donor_id,n_genes_k,n_counts_k
index,,,,,,,
AAACATACAACCAC-1,781,0.030178,2419.0,CD4 T cells,donor_1,0.781,2.419
AAACATTGAGCTAC-1,1352,0.037936,4903.0,B cells,donor_2,1.352,4.903
AAACATTGATCAGC-1,1131,0.008897,3147.0,CD4 T cells,donor_3,1.131,3.147
AAACCGTGCTTCCG-1,960,0.017431,2639.0,CD14+ Monocytes,donor_1,0.960,2.639
AAACCGTGTATGCG-1,522,0.012245,980.0,NK cells,donor_2,0.522,0.980


`summarize` reduces to a pandas DataFrame rather than an AnnData object.

In [22]:
adata.summarize(
    cells=ap.n(),
    median_genes=ap.median(obs.n_genes),
    mean_NKG7=ap.mean(raw.NKG7),
    by=obs.louvain,
).round(3)

,louvain,cells,median_genes,mean_NKG7
0,CD4 T cells,1144,809.0,0.119
1,B cells,342,677.0,0.061
2,CD14+ Monocytes,480,859.0,0.162
3,NK cells,154,890.0,3.102
4,CD8 T cells,316,824.5,1.877
5,FCGR3A+ Monocytes,150,1272.0,0.303
6,Dendritic cells,37,1544.0,0.303
7,Megakaryocytes,15,364.0,0.285


## Reuse groups

In [23]:
grouped = adata.group_by(obs.louvain)
grouped.group_keys()

,louvain
0,CD4 T cells
1,B cells
2,CD14+ Monocytes
3,NK cells
4,CD8 T cells
5,FCGR3A+ Monocytes
6,Dendritic cells
7,Megakaryocytes


In [24]:
key, group = next(iter(grouped))
key, group.n_obs

({'louvain': 'CD4 T cells'}, 1144)

In [25]:
grouped.count(sort=True)

,louvain,n
0,CD4 T cells,1144
1,CD14+ Monocytes,480
2,B cells,342
3,CD8 T cells,316
4,NK cells,154
5,FCGR3A+ Monocytes,150
6,Dendritic cells,37
7,Megakaryocytes,15


Grouped `mutate` evaluates window helpers such as `row_number` within each group.

In [26]:
grouped.mutate(
    within_type=ap.row_number(),
    depth_rank=ap.min_rank(obs.n_counts, descending=True),
).ungroup().obs.head()

,n_genes,percent_mito,n_counts,louvain,donor_id,within_type,depth_rank
index,,,,,,,
AAACATACAACCAC-1,781,0.030178,2419.0,CD4 T cells,donor_1,1,497.0
AAACATTGAGCTAC-1,1352,0.037936,4903.0,B cells,donor_2,1,9.0
AAACATTGATCAGC-1,1131,0.008897,3147.0,CD4 T cells,donor_3,2,108.0
AAACCGTGCTTCCG-1,960,0.017431,2639.0,CD14+ Monocytes,donor_1,1,186.0
AAACCGTGTATGCG-1,522,0.012245,980.0,NK cells,donor_2,1,153.0


In [27]:
grouped.slice_max(obs.n_counts, n=1).ungroup().obs[["louvain", "n_counts"]]

,louvain,n_counts
index,,
CATACTTGGGTTAC-1,CD4 T cells,7167.0
CAGGTTGAGGATCT-1,B cells,8011.0
AACCTACTGTGAGG-1,CD14+ Monocytes,5682.0
CAGTTTACACACGT-1,NK cells,5343.0
GCCTCAACTCTTTG-1,CD8 T cells,5981.0
ATTTAGGAACCATG-1,FCGR3A+ Monocytes,5677.0
GGGCCAACCTTGGA-1,Dendritic cells,8415.0
ACGAACTGGCTATG-1,Megakaryocytes,8875.0


## Build tidy tables

In [28]:
adata.to_wide(
    obs.louvain,
    raw.MS4A1,
    raw.CD79A,
    X_pca[0],
    X_pca[1],
    max_matrix_values=4 * adata.n_obs,
).head()

,louvain,raw_MS4A1,raw_CD79A,X_pca_0,X_pca_1
index,,,,,
AAACATACAACCAC-1,CD4 T cells,0.0,0.0,5.556233,0.257714
AAACATTGAGCTAC-1,B cells,1.94591,1.386294,7.209530,7.481985
AAACATTGATCAGC-1,CD4 T cells,0.0,0.0,2.694438,-1.583658
AAACCGTGCTTCCG-1,CD14+ Monocytes,0.0,0.0,-10.143295,-1.368530
AAACCGTGTATGCG-1,NK cells,0.0,0.0,-1.112816,-8.152788


In [29]:
long = adata.join_features(raw.MS4A1, raw.CD79A, shape="long", max_matrix_values=2 * adata.n_obs)
long.head()

,.cell,.feature,.abundance_raw,n_genes,percent_mito,n_counts,louvain,donor_id
0,AAACATACAACCAC-1,MS4A1,0.0,781,0.030178,2419.0,CD4 T cells,donor_1
1,AAACATACAACCAC-1,CD79A,0.0,781,0.030178,2419.0,CD4 T cells,donor_1
2,AAACATTGAGCTAC-1,MS4A1,1.94591,1352,0.037936,4903.0,B cells,donor_2
3,AAACATTGAGCTAC-1,CD79A,1.386294,1352,0.037936,4903.0,B cells,donor_2
4,AAACATTGATCAGC-1,MS4A1,0.0,1131,0.008897,3147.0,CD4 T cells,donor_3


In [30]:
ap.tidyr.pivot_wider(long, id_cols=[".cell", "louvain"], names_from=".feature", values_from=".abundance_raw").head()

,.cell,louvain,CD79A,MS4A1
0,AAACATACAACCAC-1,CD4 T cells,0.0,0.0
1,AAACATTGAGCTAC-1,B cells,1.386294,1.94591
2,AAACATTGATCAGC-1,CD4 T cells,0.0,0.0
3,AAACCGTGCTTCCG-1,CD14+ Monocytes,0.0,0.0
4,AAACCGTGTATGCG-1,NK cells,0.0,0.0


PCA loadings live in `varm` and come out through `pull`.

In [31]:
pd.concat([adata.pull(PCs[0]), adata.pull(PCs[1])], axis=1).head().round(3)

,0,1
index,,
TNFRSF4,-0.026,0.003
CPSF3L,-0.008,0.009
ATAD3C,-0.003,0.003
C1orf86,0.011,-0.000
RER1,0.014,0.027


The pandas-only helpers in `ap.tidyr` cover common tidyr-style list-column and string workflows.

In [32]:
nested = ap.tidyr.nest(adata.obs.reset_index(), by="louvain", columns=["index", "n_genes", "n_counts"])
nested

,louvain,data
0,CD4 T cells,index n_genes n_counts 0 ...
1,B cells,index n_genes n_counts 0 ...
2,CD14+ Monocytes,index n_genes n_counts 0 ...
3,NK cells,index n_genes n_counts 0 ...
4,CD8 T cells,index n_genes n_counts 0 ...
5,FCGR3A+ Monocytes,index n_genes n_counts 0 ...
6,Dendritic cells,index n_genes n_counts 0 AA...
7,Megakaryocytes,index n_genes n_counts 0 AC...


In [33]:
ap.tidyr.unnest(nested, "data").head()

,louvain,index,n_genes,n_counts
0,CD4 T cells,AAACATACAACCAC-1,781,2419.0
1,CD4 T cells,AAACATTGATCAGC-1,1131,3147.0
2,CD4 T cells,AAACGCTGTAGCCA-1,533,1275.0
3,CD4 T cells,AAACTTGATCCAGA-1,751,2388.0
4,CD4 T cells,AAAGAGACGAGATA-1,866,2410.0


In [34]:
panel = pd.DataFrame({"panel": ["b_cell", "cytotoxic"], "markers": ["MS4A1,CD79A", "NKG7,GNLY"]})
ap.tidyr.separate_rows(panel, "markers")

,panel,markers
0,b_cell,MS4A1
1,b_cell,CD79A
2,cytotoxic,NKG7
3,cytotoxic,GNLY


## Compose with project code

In [35]:
def cell_type_counts(data):
    return data.count(obs.louvain, sort=True)


adata.pipe(cell_type_counts)

,louvain,n
0,CD4 T cells,1144
1,CD14+ Monocytes,480
2,B cells,342
3,CD8 T cells,316
4,NK cells,154
5,FCGR3A+ Monocytes,150
6,Dendritic cells,37
7,Megakaryocytes,15


`annplyr` verbs chain naturally before the final `pipe` call.

In [36]:
(
    adata.filter(obs.percent_mito < 0.03)
    .mutate(b_score=(raw.MS4A1 + raw.CD79A) / 2)
    .pipe(lambda data: data.summarize(mean_b_score=ap.mean(obs.b_score), by=obs.louvain))
    .round(3)
)

,louvain,mean_b_score
0,CD4 T cells,0.034
1,CD14+ Monocytes,0.032
2,NK cells,0.028
3,CD8 T cells,0.023
4,FCGR3A+ Monocytes,0.051
5,B cells,1.208
6,Dendritic cells,0.111
7,Megakaryocytes,0.029
